Dataset: CoNLL-2003  
Task: Token Classification (Chunking)

Label Categories:
- B-NP (Beginning of Noun Phrase)
- I-NP (Inside Noun Phrase)
- B-VP (Verb Phrase)
- I-VP
- O (Outside any phrase)

In [3]:
!pip install transformers -q

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

In [5]:
sentences = [
    ["John", "works", "at", "Google"],
    ["She", "lives", "in", "California"],
    ["Microsoft", "is", "a", "company"]
]

labels = [
    ["B-NP", "B-VP", "O", "B-NP"],
    ["B-NP", "B-VP", "O", "B-NP"],
    ["B-NP", "O", "O", "O"]
]

We tokenize sentences using BERT tokenizer.
We align labels with tokens.
Special tokens and subwords are handled using -100.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [7]:
label_list = ["B-NP", "B-VP", "O"]

label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

In [8]:
def tokenize_and_align(sentence, label):
    tokenized = tokenizer(sentence, is_split_into_words=True, truncation=True)

    word_ids = tokenized.word_ids()
    aligned_labels = []

    previous_word_idx = None
    for word_idx in word_ids:
        if word_idx is None:
            aligned_labels.append(-100)
        elif word_idx != previous_word_idx:
            aligned_labels.append(label2id[label[word_idx]])
        else:
            aligned_labels.append(-100)
        previous_word_idx = word_idx

    tokenized["labels"] = aligned_labels
    return tokenized

dataset = [tokenize_and_align(s, l) for s, l in zip(sentences, labels)]

After preprocessing we get:
- input_ids
- attention_mask
- labels

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

In [10]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    num_train_epochs=3
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

trainer.train()

Model performance is evaluated using Precision, Recall, and F1 Score.
Due to small dataset, values may not be high but demonstrate working pipeline.

In [12]:
sentence = "John works at Google"
inputs = tokenizer(sentence, return_tensors="pt")
outputs = model(**inputs)

predictions = outputs.logits.argmax(dim=2)
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

for token, pred in zip(tokens, predictions[0]):
    print(token, id2label[pred.item()])

[CLS] B-VP
john B-NP
works B-NP
at O
google B-NP
[SEP] O


POS Tagging assigns grammatical labels to each word.
Chunking groups words into meaningful phrases.

POS tagging is easier as it works at word level.
Chunking is more complex as it identifies phrases.

Challenges:
- Handling subwords
- Label alignment
- Runtime issues

Observations:
- Transformer models understand context well
- Even small datasets can demonstrate pipeline